# Alpaca-py stock trading basic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alpacahq/alpaca-py/blob/master/examples/stocks/stocks-trading-basic.ipynb)

- This notebook shows how to use alpaca-py with stock trading API endpoints
- Please use ``paper account``. Please ``DO NOT`` use this notebook with live account. In this notebook, we place orders for stocks as an example.

In [2]:
# Please change the following to your own PAPER api key and secret
# or set them as environment variables (ALPACA_API_KEY, ALPACA_SECRET_KEY).
# You can get them from https://alpaca.markets/

api_key = None
secret_key = None

#### We use paper environment for this example ####
paper = True # Please do not modify this. This example is for paper trading only.
####

# Below are the variables for development this documents
# Please do not change these variables
trade_api_url = None
trade_api_wss = None
data_api_url = None
stream_data_wss = None

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

if api_key is None:
    api_key = os.environ.get('ALPACA_API_KEY')

if secret_key is None:
    secret_key = os.environ.get('ALPACA_SECRET_KEY')

In [4]:
# install alpaca-py if it is not available
try:
    import alpaca
except ImportError:
    !python3 -m pip install alpaca-py
    import alpaca

In [5]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from alpaca.trading.client import TradingClient
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.historical.corporate_actions import CorporateActionsClient
from alpaca.data.historical.stock import StockHistoricalDataClient
from alpaca.trading.stream import TradingStream
from alpaca.data.live.stock import StockDataStream

from alpaca.data.requests import (
    CorporateActionsRequest,
    StockBarsRequest,
    StockQuotesRequest,
    StockTradesRequest,
)
from alpaca.trading.requests import (
    ClosePositionRequest,
    GetAssetsRequest,
    GetOrdersRequest,
    LimitOrderRequest,
    MarketOrderRequest,
    StopLimitOrderRequest,
    StopLossRequest,
    StopOrderRequest,
    TakeProfitRequest,
    TrailingStopOrderRequest,
)
from alpaca.trading.enums import (
    AssetExchange,
    AssetStatus,
    OrderClass,
    OrderSide,
    OrderType,
    QueryOrderStatus,
    TimeInForce,
)

In [6]:
# to run async code in jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [7]:
# check version of alpaca-py
alpaca.__version__

'0.43.2'

# Trading Client

In [12]:
# setup clients
trade_client = TradingClient(api_key=api_key, secret_key=secret_key, paper=paper, url_override=trade_api_url)

In [13]:
# check trading account
# You can check definition of each field in the following documents
# ref. https://docs.alpaca.markets/docs/account-plans
# ref. https://docs.alpaca.markets/reference/getaccount-1
acct = trade_client.get_account()
acct

{   'account_blocked': False,
    'account_number': 'PA3ZPB2W362P',
    'accrued_fees': '0',
    'buying_power': '191823.1',
    'cash': '102961.55',
    'created_at': datetime.datetime(2025, 2, 7, 1, 47, 10, 815394, tzinfo=TzInfo(UTC)),
    'crypto_status': <AccountStatus.ACTIVE: 'ACTIVE'>,
    'currency': 'USD',
    'daytrade_count': 0,
    'daytrading_buying_power': '0',
    'equity': '102632.55',
    'id': UUID('94ccd725-6370-4308-b880-31e14648310a'),
    'initial_margin': '7050',
    'last_equity': '102638.55',
    'last_maintenance_margin': '7050',
    'long_market_value': '0',
    'maintenance_margin': '7050',
    'multiplier': '2',
    'non_marginable_buying_power': '95911.55',
    'options_approved_level': 3,
    'options_buying_power': '95911.55',
    'options_trading_level': 3,
    'pattern_day_trader': False,
    'pending_transfer_in': None,
    'pending_transfer_out': None,
    'portfolio_value': '102632.55',
    'regt_buying_power': '191823.1',
    'short_market_value': '

In [19]:
# check account configuration
# ref. https://docs.alpaca.markets/reference/getaccountconfig-1
acct_config = trade_client.get_account_configurations()
acct_config

{   'dtbp_check': <DTBPCheck.ENTRY: 'entry'>,
    'fractional_trading': False,
    'max_margin_multiplier': '4',
    'max_options_trading_level': None,
    'no_shorting': False,
    'pdt_check': <PDTCheck.ENTRY: 'entry'>,
    'ptp_no_exception_entry': False,
    'suspend_trade': False,
    'trade_confirm_email': <TradeConfirmationEmail.ALL: 'all'>}

In [18]:
# set account configuration
# ref. https://docs.alpaca.markets/reference/patchaccountconfig-1
req = acct_config
req.fractional_trading = not req.fractional_trading # toggle fractional trading
acct_config_new = trade_client.set_account_configurations(req)
display(acct_config_new)

# revert changes
req = acct_config_new
req.fractional_trading = not req.fractional_trading # toggle fractional trading
acct_config_reverted = trade_client.set_account_configurations(req)
display(acct_config_reverted)

{   'dtbp_check': <DTBPCheck.ENTRY: 'entry'>,
    'fractional_trading': True,
    'max_margin_multiplier': '4',
    'max_options_trading_level': None,
    'no_shorting': False,
    'pdt_check': <PDTCheck.ENTRY: 'entry'>,
    'ptp_no_exception_entry': False,
    'suspend_trade': False,
    'trade_confirm_email': <TradeConfirmationEmail.ALL: 'all'>}

{   'dtbp_check': <DTBPCheck.ENTRY: 'entry'>,
    'fractional_trading': False,
    'max_margin_multiplier': '4',
    'max_options_trading_level': None,
    'no_shorting': False,
    'pdt_check': <PDTCheck.ENTRY: 'entry'>,
    'ptp_no_exception_entry': False,
    'suspend_trade': False,
    'trade_confirm_email': <TradeConfirmationEmail.ALL: 'all'>}

In [22]:
# get list of assets which are us_equity (default), active, and in NASDAQ
# ref. https://docs.alpaca.markets/reference/get-v2-assets-1
req = GetAssetsRequest(
  # asset_class=AssetClass.US_EQUITY,  # default asset_class is us_equity
  status=AssetStatus.ACTIVE,
  exchange=AssetExchange.NASDAQ,
)
assets = trade_client.get_all_assets(req)
assets[:2]

[{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'attributes': [],
     'easy_to_borrow': False,
     'exchange': <AssetExchange.NASDAQ: 'NASDAQ'>,
     'fractionable': False,
     'id': UUID('7f82dad7-a655-475e-866a-21110b263e9f'),
     'maintenance_margin_requirement': 100.0,
     'marginable': False,
     'min_order_size': None,
     'min_trade_increment': None,
     'name': 'METAL SKY STAR ACQUISITION CORP Warrant   2029-03-31',
     'price_increment': None,
     'shortable': False,
     'status': <AssetStatus.ACTIVE: 'active'>,
     'symbol': 'MSSWF',
     'tradable': False},
 {   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'attributes': ['overnight_halted'],
     'easy_to_borrow': False,
     'exchange': <AssetExchange.NASDAQ: 'NASDAQ'>,
     'fractionable': False,
     'id': UUID('f8815179-565c-4209-a079-389ea15a4370'),
     'maintenance_margin_requirement': 100.0,
     'marginable': False,
     'min_order_size': None,
     'min_trade_increment': None,


### Orders

In [23]:
# we will place orders which Alapca trading platform supports
# - order classes: simple, bracket, oco, oto
# - order types: market, limit, stop, stop_limit, trailing_stop
#
# please refer to the following documents for more details
# ref. https://docs.alpaca.markets/docs/orders-at-alpaca
# ref. https://docs.alpaca.markets/reference/postorder
#
# we will also use fractional trading capability of Alpaca trading platform in this example
# ref. https://docs.alpaca.markets/docs/fractional-trading

# we will place orders for symbol: SPY in this example
symbol = "SPY"

In [24]:
# simple, market order, fractional qty
# Alpaca trading platform support fractional trading by default
# you can specify:
# fractional qty (e.g. 0.01 qty) in the order request (which is shown in this example)
# or notional value (e.g. 100 USD) (which is in the next example)
#
# If you have an error of `qty must be integer`,
# please try to `Reset Account` of your paper account via the Alpaca Trading API dashboard
req = MarketOrderRequest(
    symbol = symbol,
    qty = 5.5,
    side = OrderSide.BUY,
    type = OrderType.MARKET,
    time_in_force = TimeInForce.DAY,
)
res = trade_client.submit_order(req)
res

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'canceled_at': None,
    'client_order_id': 'd7ed188c-9134-422c-908b-0a9f0f2a74dc',
    'created_at': datetime.datetime(2026, 2, 6, 11, 24, 48, 761255, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 2, 6, 21, 0, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('9c88cee1-3df5-4f4e-975d-9f62ac41cf7d'),
    'legs': None,
    'limit_price': None,
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.MARKET: 'market'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '5.5',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderStat

In [25]:
# simple, market order, notional
# Alpaca trading platform support fractional trading by default
# you can specify:
# fractional qty (e.g. 0.01 qty) in the order request (which is in the example above)
# or notional value (e.g. 100 USD) (which is in this example)
req = MarketOrderRequest(
    symbol = symbol,
    notional = 1.11,  # notional is specified in USD, here we specify $1.11
    side = OrderSide.BUY,
    type = OrderType.MARKET,
    time_in_force = TimeInForce.DAY,
)
res = trade_client.submit_order(req)
res

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'canceled_at': None,
    'client_order_id': 'fb2c8c7a-3a83-45bf-a851-e56fefb50b27',
    'created_at': datetime.datetime(2026, 2, 6, 11, 56, 36, 566794, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 2, 6, 21, 0, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('6bd5b01e-f1a1-44b6-a10f-ab137e30b55c'),
    'legs': None,
    'limit_price': None,
    'notional': '1.11',
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.MARKET: 'market'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': None,
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderSta

In [26]:
# simple, limit order, fractional qty
req = LimitOrderRequest(
    symbol = symbol,
    qty = 0.01,
    limit_price = 550.25,
    side = OrderSide.BUY,
    type = OrderType.LIMIT,
    time_in_force = TimeInForce.DAY,
)
res = trade_client.submit_order(req)
res

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'canceled_at': None,
    'client_order_id': 'c52c1f03-bf0c-4185-aea2-3cfc482a71c9',
    'created_at': datetime.datetime(2026, 2, 6, 11, 57, 6, 479846, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 2, 6, 21, 0, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('89c628c9-1d72-4c9e-ab5d-aaff0ca68e83'),
    'legs': None,
    'limit_price': '550.25',
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.LIMIT: 'limit'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '0.01',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderSt

In [27]:
# stop order
req = StopOrderRequest(
                    symbol = symbol,
                    qty = 1,
                    side = OrderSide.BUY,
                    time_in_force = TimeInForce.GTC,
                    stop_price = 600
                    )

res = trade_client.submit_order(req)
res

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'canceled_at': None,
    'client_order_id': '66a128e3-d1d8-405a-8969-ea1732a1aece',
    'created_at': datetime.datetime(2026, 2, 6, 11, 57, 31, 386560, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 5, 7, 20, 0, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('b2f964f7-92ed-44c9-ac73-9cbf82b8ac71'),
    'legs': None,
    'limit_price': None,
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.STOP: 'stop'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '1',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderStatus.PEN

In [28]:
# stop limit order
req = StopLimitOrderRequest(
                    symbol = symbol,
                    qty = 1,
                    side = OrderSide.BUY,
                    time_in_force = TimeInForce.GTC,
                    limit_price = 550,
                    stop_price = 600
                    )

res = trade_client.submit_order(req)
res

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'canceled_at': None,
    'client_order_id': '9e886816-a491-4b8d-8c9b-f4b0f1be1dd3',
    'created_at': datetime.datetime(2026, 2, 6, 11, 57, 56, 109033, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 5, 7, 20, 0, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('c4266390-ecb9-4468-ae51-f1bc4e3db4e9'),
    'legs': None,
    'limit_price': '550',
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.STOP_LIMIT: 'stop_limit'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '1',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <Or

In [33]:
# bracket order with both stop loss and take profit
req = MarketOrderRequest(
                    symbol = symbol,
                    qty = 5,
                    side = OrderSide.BUY,
                    time_in_force = TimeInForce.DAY,
                    order_class = OrderClass.BRACKET,
                    take_profit = TakeProfitRequest(limit_price=710),
                    stop_loss = StopLossRequest(stop_price=300)
)
res = trade_client.submit_order(req)
res

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'canceled_at': None,
    'client_order_id': '64397d29-bb60-461f-b467-2d9887f854ac',
    'created_at': datetime.datetime(2026, 2, 7, 1, 30, 7, 247755, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 2, 9, 21, 0, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('529ec77c-0c6f-4ab4-a911-2ce6e96631cb'),
    'legs': [   {   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
                    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
                    'canceled_at': None,
                    'client_order_id': '6490e7e4-108c-495c-920a-fecbf9d846c6',
                    'created_at': datetime.datetime(2026, 2, 7, 1, 30, 7, 247755, tzinfo=TzInfo(UTC)),
                    'expired_at':

In [34]:
# oto order with stop loss
req = LimitOrderRequest(
                    symbol = symbol,
                    qty = 1,
                    limit_price = 500,
                    side = OrderSide.BUY,
                    time_in_force = TimeInForce.DAY,
                    Class = OrderClass.OTO,
                    stop_loss = StopLossRequest(stop_price = 300)
                    )

res = trade_client.submit_order(req)
res

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'canceled_at': None,
    'client_order_id': '5feebf32-11f6-4587-95d9-f0fe75f1c7ec',
    'created_at': datetime.datetime(2026, 2, 7, 1, 30, 38, 33842, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 2, 9, 21, 0, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('8ad55083-da4b-47ec-ad32-e5bd4edf1504'),
    'legs': None,
    'limit_price': '500',
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.LIMIT: 'limit'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '1',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderStatus.AC

In [35]:
# oco limit order
req = LimitOrderRequest(
                    symbol = symbol,
                    qty = 1,
                    limit_price = 500,
                    side = OrderSide.BUY,
                    time_in_force = TimeInForce.DAY,
                    Class = OrderClass.OCO
                    )

res = trade_client.submit_order(req)
res

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'canceled_at': None,
    'client_order_id': 'd5bc7fde-c2f1-43f5-a0ed-58a336b2372c',
    'created_at': datetime.datetime(2026, 2, 7, 1, 30, 57, 415780, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 2, 9, 21, 0, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('d3674920-b0da-4169-babe-365559ad9d39'),
    'legs': None,
    'limit_price': '500',
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.LIMIT: 'limit'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '1',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderStatus.A

In [36]:
# trailing stop order
req = TrailingStopOrderRequest(
                    symbol = symbol,
                    qty = 1,
                    side = OrderSide.SELL,
                    time_in_force = TimeInForce.GTC,
                    trail_percent = 0.20 # you can also use trail_price instead of trail_percent
                    )

res = trade_client.submit_order(req)
res

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'canceled_at': None,
    'client_order_id': 'ac10a285-94e0-4883-93e5-0b4034057a60',
    'created_at': datetime.datetime(2026, 2, 7, 1, 31, 26, 567064, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 5, 8, 20, 0, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': '690.62',
    'id': UUID('db48fb03-1f17-4fc4-b619-54b9a0d84baa'),
    'legs': None,
    'limit_price': None,
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.TRAILING_STOP: 'trailing_stop'>,
    'position_intent': <PositionIntent.SELL_TO_CLOSE: 'sell_to_close'>,
    'qty': '1',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.SELL: 'sell'>,
   

In [37]:
# get a list of orders including closed (e.g. filled) orders by specifying symbol
req = GetOrdersRequest(
    status = QueryOrderStatus.ALL,
    symbols = [symbol]
)
orders = trade_client.get_orders(req)
orders

[{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
     'canceled_at': None,
     'client_order_id': 'ac10a285-94e0-4883-93e5-0b4034057a60',
     'created_at': datetime.datetime(2026, 2, 7, 1, 31, 26, 567064, tzinfo=TzInfo(UTC)),
     'expired_at': None,
     'expires_at': datetime.datetime(2026, 5, 8, 20, 0, tzinfo=TzInfo(UTC)),
     'extended_hours': False,
     'failed_at': None,
     'filled_at': None,
     'filled_avg_price': None,
     'filled_qty': '0',
     'hwm': '690.62',
     'id': UUID('db48fb03-1f17-4fc4-b619-54b9a0d84baa'),
     'legs': None,
     'limit_price': None,
     'notional': None,
     'order_class': <OrderClass.SIMPLE: 'simple'>,
     'order_type': <OrderType.TRAILING_STOP: 'trailing_stop'>,
     'position_intent': <PositionIntent.SELL_TO_CLOSE: 'sell_to_close'>,
     'qty': '1',
     'ratio_qty': None,
     'replaced_at': None,
     'replaced_by': None,
     'replaces': None,
     'side': <Or

In [40]:
# see all open orders
req = GetOrdersRequest(
    status = QueryOrderStatus.OPEN,
    symbols = [symbol]
)
open_orders = trade_client.get_orders(req)
open_orders

[{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
     'canceled_at': None,
     'client_order_id': '9e886816-a491-4b8d-8c9b-f4b0f1be1dd3',
     'created_at': datetime.datetime(2026, 2, 6, 11, 57, 56, 109033, tzinfo=TzInfo(UTC)),
     'expired_at': None,
     'expires_at': datetime.datetime(2026, 5, 7, 20, 0, tzinfo=TzInfo(UTC)),
     'extended_hours': False,
     'failed_at': None,
     'filled_at': None,
     'filled_avg_price': None,
     'filled_qty': '0',
     'hwm': None,
     'id': UUID('c4266390-ecb9-4468-ae51-f1bc4e3db4e9'),
     'legs': None,
     'limit_price': '550',
     'notional': None,
     'order_class': <OrderClass.SIMPLE: 'simple'>,
     'order_type': <OrderType.STOP_LIMIT: 'stop_limit'>,
     'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
     'qty': '1',
     'ratio_qty': None,
     'replaced_at': None,
     'replaced_by': None,
     'replaces': None,
     'side': <OrderSide.BUY:

In [39]:
# cancel all open orders
trade_client.cancel_orders()

[{   'body': None,
     'id': UUID('529ec77c-0c6f-4ab4-a911-2ce6e96631cb'),
     'status': 200},
 {   'body': None,
     'id': UUID('8ad55083-da4b-47ec-ad32-e5bd4edf1504'),
     'status': 200},
 {   'body': None,
     'id': UUID('b2f964f7-92ed-44c9-ac73-9cbf82b8ac71'),
     'status': 200},
 {   'body': None,
     'id': UUID('c4266390-ecb9-4468-ae51-f1bc4e3db4e9'),
     'status': 200},
 {   'body': None,
     'id': UUID('d3674920-b0da-4169-babe-365559ad9d39'),
     'status': 200},
 {   'body': None,
     'id': UUID('db48fb03-1f17-4fc4-b619-54b9a0d84baa'),
     'status': 200}]

### Positions

In [41]:
# get all open positions
# ref. https://docs.alpaca.markets/reference/getallopenpositions-1
positions = trade_client.get_all_positions()
positions

[{   'asset_class': <AssetClass.US_OPTION: 'us_option'>,
     'asset_id': UUID('97bb1b04-2966-4e4f-b70d-38b4c54cc1f1'),
     'asset_marginable': True,
     'avg_entry_price': '0.12',
     'avg_entry_swap_rate': None,
     'change_today': '-0.66667',
     'cost_basis': '-12',
     'current_price': '0.01',
     'exchange': <AssetExchange.EMPTY: ''>,
     'lastday_price': '0.03',
     'market_value': '-1',
     'qty': '-1',
     'qty_available': '-1',
     'side': <PositionSide.SHORT: 'short'>,
     'swap_rate': None,
     'symbol': 'AAL260206P00013500',
     'unrealized_intraday_pl': '2',
     'unrealized_intraday_plpc': '0.66667',
     'unrealized_pl': '11',
     'unrealized_plpc': '0.91667',
     'usd': None},
 {   'asset_class': <AssetClass.US_OPTION: 'us_option'>,
     'asset_id': UUID('01e7fb9b-3569-4a0e-9dbc-330c908a4818'),
     'asset_marginable': True,
     'avg_entry_price': '1.65',
     'avg_entry_swap_rate': None,
     'change_today': '-0.69375',
     'cost_basis': '-165',
   

In [42]:
# get positions by symbol
# ref. https://docs.alpaca.markets/reference/getopenposition-1
position = trade_client.get_open_position(symbol_or_asset_id=symbol)
position


{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'asset_marginable': True,
    'avg_entry_price': '681.489082',
    'avg_entry_swap_rate': None,
    'change_today': '0.01918',
    'cost_basis': '3749.29',
    'current_price': '690.62',
    'exchange': <AssetExchange.ARCA: 'ARCA'>,
    'lastday_price': '677.62',
    'market_value': '3799.524786',
    'qty': '5.501614181',
    'qty_available': '5.501614181',
    'side': <PositionSide.LONG: 'long'>,
    'swap_rate': None,
    'symbol': 'SPY',
    'unrealized_intraday_pl': '71.520985',
    'unrealized_intraday_plpc': '0.01918',
    'unrealized_pl': '50.234786',
    'unrealized_plpc': '0.0134',
    'usd': None}

In [43]:
# get positions by asset_id
trade_client.get_open_position(symbol_or_asset_id=position.asset_id)

{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'asset_marginable': True,
    'avg_entry_price': '681.489082',
    'avg_entry_swap_rate': None,
    'change_today': '0.01918',
    'cost_basis': '3749.29',
    'current_price': '690.62',
    'exchange': <AssetExchange.ARCA: 'ARCA'>,
    'lastday_price': '677.62',
    'market_value': '3799.524786',
    'qty': '5.501614181',
    'qty_available': '5.501614181',
    'side': <PositionSide.LONG: 'long'>,
    'swap_rate': None,
    'symbol': 'SPY',
    'unrealized_intraday_pl': '71.520985',
    'unrealized_intraday_plpc': '0.01918',
    'unrealized_pl': '50.234786',
    'unrealized_plpc': '0.0134',
    'usd': None}

In [ ]:
# close the position with specifying qty
# ref. https://docs.alpaca.markets/reference/deleteopenposition-1
trade_client.close_position(
    symbol_or_asset_id = symbol,
    close_options = ClosePositionRequest(
        qty = "0.01",
    )
)

APIError: {"code":40310000,"existing_order_id":"b2f964f7-92ed-44c9-ac73-9cbf82b8ac71","message":"potential wash trade detected. use complex orders","reject_reason":"opposite side market/stop order exists"}

# Trade Update (Stream)

With TradingStream client, you can get updates about trades

You can open this notebook in another window and run below cell to check trade updates.

In [46]:
# subscribe trade updates
trade_stream_client = TradingStream(api_key, secret_key, paper=paper, url_override = trade_api_wss)

async def trade_updates_handler(data):
    print(data)

trade_stream_client.subscribe_trade_updates(trade_updates_handler)
trade_stream_client.run()

keyboard interrupt, bye


# Market Data (Historical)

In [47]:
# setup stock historical data client
stock_historical_data_client = StockHistoricalDataClient(api_key, secret_key, url_override = data_api_url)

In [48]:
# get historical bars by symbol
# ref. https://docs.alpaca.markets/reference/stockbars-1
now = datetime.now(ZoneInfo("America/New_York"))
req = StockBarsRequest(
    symbol_or_symbols = [symbol],
    timeframe=TimeFrame(amount = 1, unit = TimeFrameUnit.Hour), # specify timeframe
    start = now - timedelta(days = 5),                          # specify start datetime, default=the beginning of the current day.
    # end_date=None,                                        # specify end datetime, default=now
    limit = 2,                                               # specify limit
)
stock_historical_data_client.get_stock_bars(req).df

open    high     low   close    volume  \
symbol timestamp                                                             
SPY    2026-02-02 09:00:00+00:00  687.00  688.13  686.08  687.78  136815.0   
       2026-02-02 10:00:00+00:00  687.85  688.20  686.81  687.05   54492.0   

                                  trade_count       vwap  
symbol timestamp                                          
SPY    2026-02-02 09:00:00+00:00       2142.0  687.13991  
       2026-02-02 10:00:00+00:00       1066.0  687.42675

In [49]:
# get historical trades by symbol
req = StockTradesRequest(
    symbol_or_symbols = [symbol],
    start = now - timedelta(days = 5),                          # specify start datetime, default=the beginning of the current day.
    # end=None,                                             # specify end datetime, default=now
    limit = 2,                                                # specify limit
)
stock_historical_data_client.get_stock_trades(req).df

exchange   price  size  \
symbol timestamp                                                 
SPY    2026-02-02 09:00:00.023687+00:00        K  688.16   3.0   
       2026-02-02 09:00:00.023831+00:00        K  690.56   2.0   

                                                     id conditions tape  
symbol timestamp                                                         
SPY    2026-02-02 09:00:00.023687+00:00  52983525028149  [ , T, I]    B  
       2026-02-02 09:00:00.023831+00:00  52983525028150  [ , T, I]    B

In [50]:
# get historical quotes by symbol
req = StockQuotesRequest(
    symbol_or_symbols = [symbol],
    start = now - timedelta(days = 5),                      # specify start datetime, default=the beginning of the current day.
    # end=None,                                             # specify end datetime, default=now
    limit = 2,                                              # specify limit
)
stock_historical_data_client.get_stock_quotes(req).df

bid_price  bid_size bid_exchange  \
symbol timestamp                                                            
SPY    2026-02-02 09:00:00.002312+00:00     687.04      80.0            T   
       2026-02-02 09:00:00.002432+00:00     687.04      80.0            T   

                                         ask_price  ask_size ask_exchange  \
symbol timestamp                                                            
SPY    2026-02-02 09:00:00.002312+00:00       0.00       0.0                
       2026-02-02 09:00:00.002432+00:00     687.13      80.0            T   

                                        conditions tape  
symbol timestamp                                         
SPY    2026-02-02 09:00:00.002312+00:00        [R]    B  
       2026-02-02 09:00:00.002432+00:00        [R]    B

In [51]:
# get latest quotes by symbol
req = StockQuotesRequest(
    symbol_or_symbols = [symbol],
)
res = stock_historical_data_client.get_stock_latest_quote(req)
res

{'SPY': {   'ask_exchange': ' ',
     'ask_price': 0.0,
     'ask_size': 0.0,
     'bid_exchange': 'V',
     'bid_price': 667.95,
     'bid_size': 40.0,
     'conditions': ['R'],
     'symbol': 'SPY',
     'tape': 'B',
     'timestamp': datetime.datetime(2026, 2, 6, 21, 0, 1, 682080, tzinfo=TzInfo(UTC))}}

# Market Data (Stream)

In [ ]:
stock_data_stream_client = StockDataStream(api_key, secret_key, url_override = stream_data_wss)

async def stock_data_stream_handler(data):
    print(data)

symbols = [symbol]

stock_data_stream_client.subscribe_quotes(stock_data_stream_handler, *symbols)
stock_data_stream_client.subscribe_trades(stock_data_stream_handler, *symbols)

stock_data_stream_client.run()

# Corporate actions

In [53]:
corporate_actions_client = CorporateActionsClient(api_key, secret_key)
corporate_actions_client.get_corporate_actions(CorporateActionsRequest(
    start=datetime(2020, 1, 1),
    symbols=[symbol]
)).df

,id,symbol,cusip,rate,special,foreign,process_date,ex_date,record_date,payable_date
corporate_action_type,,,,,,,,,,
cash_dividends,e06f1a82-04b5-4b90-9c7b-06d22b720db1,SPY,78462F103,1.405559,False,False,2020-04-30,2020-03-20,2020-03-23,2020-04-30
cash_dividends,a6587d80-e08a-4b1a-aa94-857549880c59,SPY,78462F103,1.366242,False,False,2020-07-31,2020-06-19,2020-06-22,2020-07-31
cash_dividends,ffb8940e-53fe-4b3e-a475-e15e39561d3d,SPY,78462F103,1.339224,False,False,2020-10-30,2020-09-18,2020-09-21,2020-10-30
cash_dividends,4d35e709-773b-46d4-b292-34d14c46f287,SPY,78462F103,1.580000,False,False,2021-01-29,2020-12-18,2020-12-21,2021-01-29
cash_dividends,f1ddda1e-b10c-4576-8253-4d8d5a149de6,SPY,78462F103,1.277788,False,False,2021-04-30,2021-03-19,2021-03-22,2021-04-30
cash_dividends,a06ac608-4210-4e4f-94f0-4fa342a8e1fd,SPY,78462F103,1.375875,False,False,2021-07-30,2021-06-18,2021-06-21,2021-07-30
cash_dividends,d2b09118-e4c0-4b93-9f46-10b211400773,SPY,78462F103,1.428117,False,False,2021-10-29,2021-09-17,2021-09-20,2021-10-29
cash_dividends,7cdc9442-1a32-4f40-9d2a-b721edfef9e9,SPY,78462F103,1.636431,False,False,2022-01-31,2021-12-17,2021-12-20,2022-01-31
cash_dividends,f5ca31fd-da0d-4651-be6b-6ac78542259e,SPY,78462F103,1.366009,False,False,2022-04-29,2022-03-18,2022-03-21,2022-04-29


In [54]:
corporate_actions_client = CorporateActionsClient(api_key, secret_key)
corporate_actions_client.get_corporate_actions(CorporateActionsRequest(
    start=datetime(2020, 1, 1),
    symbols=['MSFT']
)).df

,id,symbol,cusip,rate,special,foreign,process_date,ex_date,record_date,payable_date,acquirer_symbol,acquirer_cusip,acquiree_symbol,acquiree_cusip,effective_date
corporate_action_type,,,,,,,,,,,,,,,
cash_dividends,b344d4bb-45f9-4109-aa02-501cd146f328,MSFT,,0.51,False,False,2020-02-19,2020-02-19,None,None,NaN,NaN,NaN,NaN,NaN
cash_dividends,f014486f-a6b1-4a7c-bbe9-38eed5bb176c,MSFT,594918104,0.51,False,False,2020-06-11,2020-05-20,2020-05-21,2020-06-11,NaN,NaN,NaN,NaN,NaN
cash_dividends,3e6a0486-e4ce-4338-838b-eb053c12e2f4,MSFT,594918104,0.51,False,False,2020-09-10,2020-08-19,2020-08-20,2020-09-10,NaN,NaN,NaN,NaN,NaN
cash_dividends,38dad563-d48a-4e48-b978-509a8250e126,MSFT,594918104,0.56,False,False,2020-12-10,2020-11-18,2020-11-19,2020-12-10,NaN,NaN,NaN,NaN,NaN
cash_dividends,b17739be-8d04-4f52-ba00-602fa08cbb9d,MSFT,594918104,0.56,False,False,2021-03-11,2021-02-17,2021-02-18,2021-03-11,NaN,NaN,NaN,NaN,NaN
cash_dividends,4bd422d6-7808-4cde-8aee-84a3a8459f8b,MSFT,594918104,0.56,False,False,2021-06-10,2021-05-19,2021-05-20,2021-06-10,NaN,NaN,NaN,NaN,NaN
cash_dividends,fff02e16-41f5-4402-9ef0-93ac431b2a0e,MSFT,594918104,0.56,False,False,2021-09-09,2021-08-18,2021-08-19,2021-09-09,NaN,NaN,NaN,NaN,NaN
cash_dividends,478ad092-b545-4a08-9022-938d2aac0fcc,MSFT,594918104,0.62,False,False,2021-12-09,2021-11-17,2021-11-18,2021-12-09,NaN,NaN,NaN,NaN,NaN
cash_dividends,25bec3b7-c360-4864-8bfe-4dfce11b94cb,MSFT,594918104,0.62,False,False,2022-03-10,2022-02-16,2022-02-17,2022-03-10,NaN,NaN,NaN,NaN,NaN
